# 🇱🇰 Sri Lanka Tourism: Agentic AI Travel Recommendation System
**Author:** Senior AI/ML Engineer  
**Stack:** Python 3.11, Pandas, Scikit-Learn, Pydantic v2, Matplotlib, FastAPI, OpenAI / Claude API

---

### Architecture Overview
This notebook demonstrates an end-to-end **Agentic AI Travel Recommendation Engine** tailored for Sri Lankan attractions. The architecture combines:
1. **Rule-based Baseline Engine:** Multi-factor scoring ($0-100$) including Bayesian-weighted ratings, Haversine distance, budget fit, category interest, and user review affinity.
2. **Machine Learning Layer:** TF-IDF text vectorization and Cosine Similarity on attraction descriptions and review sentiments for content-based similarity, with cold-start fallbacks.
3. **Autonomous Agent Loop:** Tool-calling LLM reasoning cycle that queries specialized tools, enforces strict anti-hallucination grounding rules, and validates structured JSON output with Pydantic.
4. **Multi-Turn Memory & Fallbacks:** Conversational state tracking and automatic fallback to rule-based recommendations on failure.
5. **Production Microservice:** FastAPI export (`agent_service.py`) and C# ASP.NET Core forwarding integration guide.

## 1. Setup and Configuration
We import required scientific and agent libraries, load environment variables securely using `python-dotenv`, and configure the default scoring weights and system boundaries.

In [ ]:
import os
import sys
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv

# Add current directory to path for modular imports
current_dir = Path.cwd()
if str(current_dir) not in sys.path:
    sys.path.append(str(current_dir))

# Load environment variables (.env)
load_dotenv()

# Configuration parameters & scoring weights
CONFIG = {
    "weights": {
        "interest": 0.30,
        "rating": 0.25,
        "budget": 0.15,
        "distance": 0.15,
        "popularity": 0.10,
        "history": 0.05
    },
    "bayesian_m": 3.0,          # Minimum review threshold for Bayesian weighting
    "default_user_lat": 6.9271,  # Colombo base coordinates
    "default_user_lng": 79.8612,
    "max_recommendations": 6,
    "max_agent_iterations": 5
}

print("✅ Configuration and libraries loaded successfully.")
print(f"🔑 LLM Provider Mode: {os.getenv('LLM_PROVIDER', 'auto')} (Keys configured in .env without hardcoding)")

## 2. Data Loading, Cleaning & Exploratory Data Analysis (EDA)
We generate and load 32 curated Sri Lankan attractions across 7 key travel categories (*Culture, History, Nature, Adventure, Food, Wildlife, Beaches*) and 320 synthetic reviews. We then visualize rating distributions, category balance, and daily cost distributions.

In [ ]:
from seed_data import load_dataset

# Load attractions and reviews dataset
attractions_df, reviews_df = load_dataset()

print(f"Attractions shape: {attractions_df.shape}")
print(f"Reviews shape: {reviews_df.shape}")
display(attractions_df.head(5)[['attractionId', 'name', 'category', 'activityType', 'estimatedCostPerDay', 'location']])

In [ ]:
# EDA Visualizations: Ratings, Categories, and Daily Cost
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# 1. Review Rating Distribution
reviews_df['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='#2b5c8f', edgecolor='black', alpha=0.85
)
axes[0].set_title("Distribution of Review Ratings (1-5 Stars)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Number of Reviews")
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# 2. Attractions per Category
category_counts = attractions_df['category'].value_counts()
category_counts.plot(
    kind='bar', ax=axes[1], color='#388e3c', edgecolor='black', alpha=0.85
)
axes[1].set_title("Attractions by Category in Sri Lanka", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Attraction Count")
axes[1].tick_params(axis='x', rotation=35)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

# 3. Daily Estimated Cost Distribution
axes[2].hist(attractions_df['estimatedCostPerDay'], bins=8, color='#d35400', edgecolor='black', alpha=0.85)
axes[2].set_title("Estimated Daily Cost (USD)", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Cost Per Day ($ USD)")
axes[2].set_ylabel("Frequency")
axes[2].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## 3. Rule-Based Scoring Engine (Baseline Recommender)
The baseline engine combines hard filters with a composite 0-100 score:
- **Hard Filters:** `maxBudget`, `maxDistanceKm` (Haversine from user coordinates), `minRating`, `activityType`.
- **Scoring Formula (0-100):**
  $$\text{Score} = 30\% \cdot \text{Interest} + 25\% \cdot \text{BayesianRating} + 15\% \cdot \text{BudgetFit} + 15\% \cdot \text{DistanceFit} + 10\% \cdot \text{Popularity} + 5\% \cdot \text{UserHistory}$$
- **Bayesian Weighted Rating:**
  $$WR = \frac{v}{v+m} R + \frac{m}{v+m} C$$
  (where $R$ is mean attraction rating, $v$ is review count, $m=3$ prior threshold, and $C$ is global mean rating).

In [ ]:
from scoring import score_attractions, haversine_distance, compute_attraction_aggregates

# Test baseline recommendation for an Adventure traveler with budget <= $50/day
baseline_results = score_attractions(
    attractions_df=attractions_df,
    reviews_df=reviews_df,
    interests=["Adventure", "Nature"],
    max_budget=55.0,
    max_distance_km=250.0,
    user_lat=6.9271,  # Colombo
    user_lng=79.8612,
    limit=5
)

print("=== Top 5 Baseline Rule-Based Recommendations ===")
for r in baseline_results:
    print(f"🎯 [{r['matchScore']:.1f}%] {r['name']} ({r['category']}) - ${r['estimatedCost']}/day | {r['distanceKm']}km from Colombo")
    print(f"   Reasons: {', '.join(r['matchReasons'])}")

## 4. Machine Learning Component: Content-Based Similarity & Cold-Start Fallback
We use TF-IDF vectorization across attraction categories, descriptions, and aggregated traveler reviews, followed by Cosine Similarity to recommend "similar places" to a given attraction.
When a user is brand new (cold-start), the system recommends high-confidence Bayesian-ranked destinations.

In [ ]:
from ml_recommender import ContentRecommender

# Initialize ML Content Recommender
ml_engine = ContentRecommender(attractions_df, reviews_df)

# Test 1: Find places similar to Sigiriya Ancient Rock Fortress (attractionId=1)
similar_to_sigiriya = ml_engine.get_similar_attractions(attraction_id=1, limit=4)
print("🏛️ Attractions content-similar to 'Sigiriya Ancient Rock Fortress':")
for item in similar_to_sigiriya:
    print(f" - {item['name']} ({item['category']}) | Similarity Score: {item['similarityScore']:.3f} | ${item['estimatedCost']}/day")

# Test 2: Cold-start fallback for new travelers
cold_start_recs = ml_engine.get_cold_start_popular(limit=3)
print("\n🌟 Cold-Start Fallback Recommendations (Popular + High Bayesian Rating):")
for item in cold_start_recs:
    print(f" - {item['name']} ({item['category']}) | Rating: {item['avgRating']}/5.0 ({item['reviewCount']} reviews)")

## 5. Agent Tools Definition & Pydantic Argument Validation
We define the 5 essential tools as type-hinted Python functions with docstrings:
1. `search_attractions(...)`: Multi-criteria search and ranking.
2. `get_popular_attractions(limit)`: Top rated destinations.
3. `get_review_insights(attraction_id)`: Sentiment and real review quotes.
4. `get_similar_attractions(attraction_id, limit)`: Content similarity via ML.
5. `get_user_review_history(user_id)`: Historical preferences.

Tool execution is safeguarded by a Pydantic argument dispatcher.

In [ ]:
from tools import TourismToolRegistry, LLM_TOOLS_DEFINITIONS, dispatch_tool

# Initialize Registry
registry = TourismToolRegistry(attractions_df, reviews_df)

# Test tool dispatching with validation
test_call = dispatch_tool(
    tool_name="search_attractions",
    arguments={"interests": ["Beaches"], "max_budget": 60.0, "limit": 3},
    registry=registry
)
print(f"Dispatch status: {test_call['status']}, Retrieved {test_call['count']} beach destinations:")
for a in test_call['attractions']:
    print(f" 🏖️ {a['name']} - ${a['estimatedCost']}/day (Match: {a['matchScore']}%)")

## 6. Agent Tool Loop, Grounding Rules & JSON Validation
The agent implements an autonomous tool-calling loop:
1. User prompt + optional UI filters are passed to the agent.
2. System prompt mandates **strict grounding**: only recommend attractions returned by tools; never hallucinate prices or places; max 6 results; ask 1 follow-up question if key parameters are missing.
3. Code post-check validates all IDs against the dataset.
4. If LLM call or JSON parsing fails, the agent gracefully returns the rule-based baseline recommendations.

In [ ]:
from agent import TourismAgent, AgentResponse

# Initialize Agent (Supports OpenAI, Anthropic, or Smart Mock Agent)
agent = TourismAgent(registry=registry, provider="auto")

# Run agent on an expressive natural language prompt
user_query = "I want a relaxing beach and whale watching trip under $60 per day with good seafood, starting from Colombo."
response: AgentResponse = agent.execute_agent_loop(
    user_message=user_query,
    user_id=10,
    conversation_id="demo-session-001"
)

print("\n--- AI AGENT RESPONSE ---")
print(f"📝 Summary: {response.summary}\n")
print("🎯 Recommendations:")
for rec in response.recommendations:
    print(f"  • [{rec.matchScore:.1f}%] {rec.name} (ID: {rec.attractionId}, {rec.category})")
    print(f"    Cost: ${rec.estimatedCost}/day | Location: {rec.location}")
    print(f"    Reason: {rec.reason}")

if response.followUpQuestion:
    print(f"\n❓ Follow-up Question: {response.followUpQuestion}")

## 7. Multi-Turn Conversation Memory
The agent retains recent conversation history per `conversationId`, allowing users to refine requests (e.g. "actually make it more focused on wildlife").

In [ ]:
# Multi-turn refinement on existing conversationId
follow_up_prompt = "Actually, can you swap out one beach for a nearby elephant safari under $75/day?"
turn2_response: AgentResponse = agent.execute_agent_loop(
    user_message=follow_up_prompt,
    conversation_id="demo-session-001"
)

print(f"📝 Multi-Turn Turn 2 Summary:\n{turn2_response.summary}\n")
print("🎯 Updated Recommendations:")
for rec in turn2_response.recommendations:
    print(f"  • {rec.name} ({rec.category}) - ${rec.estimatedCost}/day")

## 8. Comprehensive Evaluation & Benchmark
We evaluate the AI Agent across **8 diverse persona test prompts** (budget constraint, family trip, adventure, wildlife, food tour, vague prompt, impossible constraint, and off-topic query).
We verify:
- **Valid JSON Output**
- **No Hallucinated Attraction IDs**
- **Filters & Constraints Respected**
- **Comparison against Baseline Ranking**

In [ ]:
from tabulate import tabulate

test_cases = [
    {"id": 1, "persona": "Budget Backpacker", "prompt": "I need budget adventure and hiking under $20 a day.", "max_budget": 20.0},
    {"id": 2, "persona": "Wildlife Enthusiast", "prompt": "Where can I see wild elephants and leopards on a guided tour?", "interest": "Wildlife"},
    {"id": 3, "persona": "Family Beach Holiday", "prompt": "Quiet beach trip suitable for swimming and kids under $50/day.", "interest": "Beaches", "max_budget": 50.0},
    {"id": 4, "persona": "Cultural Heritage", "prompt": "Ancient temples, UNESCO ruins, and historical fortresses.", "interest": "History"},
    {"id": 5, "persona": "Foodie & Cooking", "prompt": "Sri Lankan street food, spice gardens, and authentic curry tours.", "interest": "Food"},
    {"id": 6, "persona": "Vague Query", "prompt": "I am planning a trip to Sri Lanka next month.", "expect_followup": True},
    {"id": 7, "persona": "Impossible Constraint", "prompt": "Find a 5-star luxury tour under $2 per day.", "expect_fallback": True},
    {"id": 8, "persona": "Off-Topic Query", "prompt": "Can you write a Python script for binary search?", "expect_empty_or_deflect": True}
]

eval_results = []
valid_ids = set(attractions_df["attractionId"].tolist())

for tc in test_cases:
    res = agent.execute_agent_loop(tc["prompt"], conversation_id=f"eval-test-{tc['id']}")
    
    # Validation checks
    has_valid_json = isinstance(res, AgentResponse)
    has_no_hallucinations = all(r.attractionId in valid_ids for r in res.recommendations)
    
    # Constraint check
    if tc.get("max_budget"):
        filters_ok = all(r.estimatedCost <= tc["max_budget"] + 5.0 for r in res.recommendations) if res.recommendations else True
    else:
        filters_ok = True
        
    passed = has_valid_json and has_no_hallucinations and filters_ok
    
    rec_names = ", ".join([r.name[:18] for r in res.recommendations[:3]]) or "(Clarification / Guidance)"
    
    eval_results.append([
        tc["id"],
        tc["persona"],
        tc["prompt"][:40] + "...",
        rec_names,
        "✅ PASS" if passed else "❌ FAIL",
        "Yes" if res.followUpQuestion else "No",
        "Yes" if res.usedFallback else "No"
    ])

print(tabulate(
    eval_results,
    headers=["ID", "Persona", "Prompt Snippet", "Top Recommendations", "Status", "Follow-Up?", "Fallback?"],
    tablefmt="grid"
))

## 9. Export as a Service: FastAPI Microservice
The reusable code has been modularized into clean Python modules:
- `seed_data.py`: Dataset loading & realistic seeding
- `scoring.py`: Baseline multi-factor scoring & Bayesian weighting
- `ml_recommender.py`: TF-IDF cosine similarity & cold-start fallback
- `tools.py`: Safe tool dispatcher & Pydantic schemas
- `agent.py`: Autonomous agent tool loop & memory
- `agent_service.py`: FastAPI server with `POST /agent` and `GET /health`

Let's test the FastAPI microservice directly via TestClient.

In [ ]:
from fastapi.testclient import TestClient
from agent_service import app

# Test FastAPI endpoints directly
with TestClient(app) as client:
    # 1. Health check
    health_res = client.get("/health")
    print("Health Check Response:", health_res.status_code, health_res.json())
    
    # 2. Agent recommendation endpoint
    agent_payload = {
        "message": "I want a culture and nature tour under $40/day in central Sri Lanka.",
        "filters": {
            "interests": ["Culture", "Nature"],
            "maxBudget": 40.0
        },
        "userId": 5,
        "conversationId": "fastapi-test-001"
    }
    agent_res = client.post("/agent", json=agent_payload)
    print("\nAgent Endpoint Status:", agent_res.status_code)
    data = agent_res.json()
    print(f"Summary: {data['summary']}")
    print(f"Returned {len(data['recommendations'])} recommendations.")

## 10. Integration Guide: ASP.NET Core & Frontend React Integration

### A. C# ASP.NET Core Forwarding Controller
In your ASP.NET Core backend (`RecommendationsController.cs`), register `HttpClient` in `Program.cs` and forward requests to the FastAPI microservice:

```csharp
// Program.cs
builder.Services.AddHttpClient("AiAgentClient", client =>
{
    client.BaseAddress = new Uri(builder.Configuration["AiService:BaseUrl"] ?? "http://localhost:8000");
    client.Timeout = TimeSpan.FromSeconds(45);
});
```

```csharp
// RecommendationsController.cs
[HttpPost("agent")]
[Authorize]
public async Task<IActionResult> AskAgent([FromBody] AgentRequestDto request)
{
    if (string.IsNullOrWhiteSpace(request.Message))
        return BadRequest(new { message = "Message is required." });

    var client = _httpClientFactory.CreateClient("AiAgentClient");
    var payload = new
    {
        message = request.Message,
        filters = request.Filters,
        userId = GetCurrentTouristId(),
        conversationId = request.ConversationId
    };

    try
    {
        var response = await client.PostAsJsonAsync("/agent", payload);
        if (response.IsSuccessStatusCode)
        {
            var result = await response.Content.ReadFromJsonAsync<AgentResponseDto>();
            return Ok(result);
        }
        
        // Fallback to internal scoring if FastAPI returns error
        _logger.LogWarning("FastAPI agent failed with status {StatusCode}, falling back.", response.StatusCode);
        return Ok(await _recommendationService.GetPersonalizedRecommendationsAsync(request.Filters ?? new(), GetCurrentTouristId()));
    }
    catch (Exception ex)
    {
        _logger.LogError(ex, "Error forwarding to AI Agent service, using fallback.");
        return Ok(await _recommendationService.GetPersonalizedRecommendationsAsync(request.Filters ?? new(), GetCurrentTouristId()));
    }
}
```

### B. Frontend React `fetch()` "Ask AI" Component

```typescript
// services/agentApi.ts
export interface AgentResponse {
  summary: string;
  recommendations: Array<{
    attractionId: number;
    name: string;
    reason: string;
    matchScore: number;
    category: string;
    imageUrl: string;
    estimatedCost: number;
    location: string;
  }>;
  followUpQuestion?: string | null;
  conversationId: string;
}

export async function askTravelAgent(message: string, conversationId?: string, filters?: any): Promise<AgentResponse> {
  const token = localStorage.getItem('token');
  const response = await fetch('/api/Recommendations/agent', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json',
      'Authorization': `Bearer ${token}`
    },
    body: JSON.stringify({ message, conversationId, filters })
  });

  if (!response.ok) {
    throw new Error(`Agent request failed with status ${response.status}`);
  }

  return response.json();
}
```